# Linguistische Annotation

Skript 01 hat Wörter als Zeichenketten behandelt. Nichts in dieser Verarbeitung wusste etwas über die deutsche Sprache: `Ministern` und `Minister` waren verschiedene Tokens, die Stoppwortliste musste von Hand bereitgestellt werden, und Casefolding (Kleinschreibung) war ein grober Ersatz dafür, zu wissen, welche Wörter Substantive sind.

In diesem Skript kommt ein **statistisches Sprachmodell** hinzu - die deutsche Pipeline von spaCy -, das jedes Token mit linguistischen Informationen annotiert, die es aus einem großen, von Hand annotierten Korpus deutscher Texte gelernt hat. Es ist ein kleines neuronales Netz, aber kein großes Sprachmodell (LLM): Es erzeugt keinen Text, sondern versieht ihn mit Labels, und jedes Label, das es vergibt, lässt sich überprüfen.

Am Ende dieses Skripts kannst du:

- eine spaCy-Pipeline ausführen und dich in ihren `Doc`-, `Token`- und `Span`-Objekten zurechtfinden;
- **Lemmata**, **Wortarten (POS)** und **Morphologie** anstelle von Tricks mit Zeichenketten verwenden;
- das Ergebnis eines **Dependenzparsings** lesen und daraus Tripel nach dem Muster "wer tut was" extrahieren;
- **Named Entities** (Eigennamen) extrahieren und einschätzen, wie weit man ihnen trauen kann;
- mit **regelbasiertem Matching** eigenes Wissen hinzufügen.

## Präambel

Wir laden alle Pakete und Module, die wir im Folgenden benötigen.

In [ ]:
# Hilfsmittel aus der Standardbibliothek
from collections import Counter, defaultdict

# Visualisierung und Datentransformation
import matplotlib.pyplot as plt
import pandas as pd

# Pakete für die linguistische Annotation
import spacy
from spacy import displacy

# Hilfsfunktionen des Kurses
import textmining as tm

# Zeitmessung
import time

# Darstellungsoptionen für Tabellen und Grafiken
pd.set_option("display.max_colwidth", 100)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

## Daten

Wir verwenden dasselbe Korpus wie in Skript 01: die Schlagzeilen von Der Standard und Süddeutscher Zeitung aus dem Jahr Juni 2015 bis Mai 2016. Die Funktion `tm.load_corpus()` aus dem Kursmodul `textmining.py` lädt es als Tabelle mit einer Zeile pro Dokument und trifft dabei alle Entscheidungen, die Skript 01 erklärt hat.

In [ ]:
# Lade das Korpus und zeige die Anzahl der Dokumente
corpus = tm.load_corpus()
print(f"{len(corpus):,} Dokumente")

Das Korpus ist dasselbe wie am Ende von Skript 01: gut 42.000 Schlagzeilen aus beiden Zeitungen, bereits bereinigt.

## Die Pipeline

spaCy bietet deutsche Modelle in mehreren Größen an. In diesem Kurs verwenden wir `de_core_news_md`: das mittlere Modell, das **Wortvektoren** mitbringt (diese verwenden wir in Skript 06) und spürbar genauer ist als das kleine. Installiere es einmalig mit `python -m spacy download de_core_news_md`.

Ein geladenes Modell ist eine **Pipeline**: ein Tokenizer, gefolgt von Komponenten, die jeweils eine Art von Annotation hinzufügen.

In [ ]:
# Lade das mittlere deutsche Modell und zeige Name, Version und Komponenten der Pipeline
nlp = spacy.load("de_core_news_md")
print(nlp.meta["name"], nlp.meta["version"])
print(nlp.pipe_names)

| Komponente | fügt hinzu |
|---|---|
| `tok2vec` | einen kontextabhängigen Vektor pro Token, den die folgenden Komponenten gemeinsam nutzen |
| `tagger` | feinkörnige Wortart-Tags (das deutsche STTS-Tagset) |
| `morphologizer` | grobe Wortart sowie morphologische Merkmale: Kasus, Numerus, Genus, Tempus |
| `parser` | den Dependenzbaum und daraus Satzgrenzen und Nominalphrasen (noun chunks) |
| `lemmatizer` | die Wörterbuchform jedes Wortes |
| `attribute_ruler` | regelbasierte Korrekturen der statistischen Tags |
| `ner` | Named Entities: Personen, Orte, Organisationen |

Wenden wir die Pipeline auf eine Zeichenkette an, erhalten wir ein `Doc`: eine Folge von `Token`-Objekten, die all diese Annotationen tragen.

In [ ]:
# Annotiere eine Beispielschlagzeile
headline = "Die Minister beschlossen in Wien neue Regeln für Spritpreise"
doc = nlp(headline)

# Zeige die Annotationen aller Tokens als Tabelle (Spaltennamen wie die spaCy-Attribute)
pd.DataFrame(
    [
        {
            "text": t.text,
            "lemma": t.lemma_,
            "pos": t.pos_,
            "tag": t.tag_,
            "morph": str(t.morph),
            "dep": t.dep_,
            "head": t.head.text,
            "stop": t.is_stop,
        }
        for t in doc
    ]
)

Diese Tabelle enthält eine Menge Information:

- **`lemma`** löst die Flexion auf: `beschlossen` -> `beschließen`; `Minister` behält seine Form, wäre aber auch das Lemma von `Ministern`.
- **`pos`** ist die grobe Wortart nach den sprachübergreifenden Tags der [Universal Dependencies](https://universaldependencies.org/u/pos/). `tag` ist das feine deutsche Tag (`NN` Substantiv, `NE` Eigenname, `VVFIN` finites Vollverb).
- **`morph`** enthält die grammatischen Merkmale, die das Deutsche an seinen Wörtern markiert. `Case=Nom|Number=Plur` bei *Minister* zeigt, dass sie das Subjekt sind.
- **`dep`** und **`head`** beschreiben die Satzstruktur (siehe Abschnitt "Dependenzparsing: Wer tut was").
- **`stop`** ist das modelleigene Kennzeichen für Stoppwörter; die NLTK-Liste aus Skript 01 brauchen wir also nicht mehr.

Beachte, dass das Modell `Spritpreise` nie zerlegt. Die Lemmatisierung löst Flexion auf, keine Komposita.

### Das ganze Korpus parsen

`nlp.pipe` verarbeitet viele Texte stapelweise (in Batches), was deutlich schneller ist, als `nlp` in einer Schleife aufzurufen. Die Hilfsfunktion `tm.parse_corpus` des Kurses baut darauf auf und speichert das Ergebnis im `DocBin`-Format von spaCy auf der Festplatte zwischen (Cache), sodass spätere Skripte dieses Parsing wiederverwenden, statt es zu wiederholen. Beim ersten Lauf dauert das Parsen der gut 40.000 Schlagzeilen einige Minuten.

In [ ]:
# Parse alle Schlagzeilen (oder lade sie aus dem Cache) und stoppe die Zeit
start = time.time()
docs = tm.parse_corpus(nlp, corpus["title"].tolist(), name="titles")
print(f"{len(docs):,} Schlagzeilen in {time.time() - start:.1f}s (schnell, wenn aus dem Cache geladen)")

Beim ersten Lauf parst spaCy alle Schlagzeilen und schreibt das Ergebnis nach `_cache/`; das dauert auf einem gewöhnlichen Laptop etwa eine halbe bis eine Minute. Bei jedem weiteren Lauf, auch in den Skripten 03 und 04, wird der Cache gelesen, was nur wenige Sekunden braucht.

## Lemmata: flektierte Formen zusammenführen

Der praktische Nutzen der Lemmatisierung zeigt sich an den Gruppen von Oberflächenformen, die auf ein einziges Lemma zusammenfallen. Jede der folgenden Gruppen wurde in Skript 01 als mehrere, voneinander unabhängige Terme gezählt.

In [ ]:
# Filter für Inhaltswörter (wird in den Skripten 03 und 04 wiederverwendet)
def is_content(token) -> bool:
    """Wörter, die sich zu zählen lohnen: keine Satzzeichen, Zahlen, Leerzeichen oder Stoppwörter.

    Bewusst nicht `token.is_alpha`: Das ist für jedes Token mit Bindestrich False,
    würde also stillschweigend `Sachsen-Anhalt`, `CDU-Chef` und jedes andere
    Kompositum mit Bindestrich löschen - viele der aussagekräftigsten Terme dieses Korpus.
    """
    return token.pos_ not in {"PUNCT", "NUM", "SPACE", "SYM", "X"} and not token.is_stop and any(
        ch.isalpha() for ch in token.text
    )


print("is_alpha bei einem Kompositum mit Bindestrich:", nlp("Sachsen-Anhalt")[0].is_alpha)

# Zähle für jedes Lemma die Oberflächenformen, die darauf abgebildet werden
forms = defaultdict(Counter)
for d in docs:
    for t in d:
        if is_content(t):
            forms[t.lemma_.casefold()][t.text.casefold()] += 1

# Sortiere nach Anzahl der Formen, dann nach Häufigkeit, und zeige die ersten 12 Lemmata
pooled = sorted(forms.items(), key=lambda kv: (-len(kv[1]), -sum(kv[1].values())))
pd.DataFrame(
    [{"Lemma": lemma, "Formen": ", ".join(f for f, _ in c.most_common()), "Anzahl": sum(c.values())}
     for lemma, c in pooled[:12]]
)

Jede Zeile ist ein Lemma mit allen Oberflächenformen, die das Modell darauf zurückgeführt hat, und der Gesamtzahl ihrer Vorkommen. Die Tabelle ist nach der Anzahl verschiedener Formen sortiert. Die Ausgabe darüber bestätigt, dass `is_alpha` für ein Kompositum mit Bindestrich `False` liefert - der Grund, warum `is_content` diese Eigenschaft nicht verwendet.

In [ ]:
# Vergleiche die Anzahl der Oberflächenformen mit der Anzahl der Lemmata
n_forms = sum(len(c) for c in forms.values())
print(f"{n_forms:,} verschiedene Oberflächenformen -> {len(forms):,} Lemmata "
      f"({1 - len(forms) / n_forms:.0%} weniger Types)")

Der Wortschatz schrumpft nur um einen bescheidenen Anteil - aber das ist der falsche Maßstab für die Lemmatisierung. Entscheidend ist, dass Evidenz, die vorher auf mehrere Formen verteilt war, jetzt zusammengeführt wird: Ein Begriff, der je einmal als *Angriff*, *Angriffe* und *Angriffen* vorkommt, ist jetzt ein Term, der dreimal vorkommt - genug, um eine Mindesthäufigkeit zu überschreiten, an der er sonst gescheitert wäre.

Die Tabelle wird von Adjektiven angeführt, weil sie mit Steigerung und Deklination die meisten Formen haben (*hoch*, *höher*, *höchste*, *hohen* …). Lemmatisierer machen aber auch Fehler, besonders bei Schlagzeilen, die im Telegrammstil geschrieben und voller Namen sind, die das Modell nie gesehen hat. Einer steht schon in der Tabelle: Das "Lemma" `--` sammelt englische und andere fremdsprachige Wörter (*new*, *the*, *love*) aus Film-, Serien- und Produktnamen, für die das Modell keine Wörterbuchform kennt. Bei einem statistischen Modell findet sich beim genauen Hinsehen fast immer ein solcher Fall.

## Wortarten: Was für eine Sprache ist eine Schlagzeile?

Wortarten ermöglichen Fragen, die sich mit Zeichenketten nicht beantworten lassen. Ein bekanntes Merkmal deutscher Schlagzeilen ist der **Nominalstil**: Die Bedeutung wird in Substantive gepackt und das Verb ganz weggelassen, wie in *"Neue Regeln für Spritpreise"*. Zeitungen unterscheiden sich darin, wie weit sie dabei gehen.

In [ ]:
# Prüfe, ob ein Dokument ein finites Verb enthält
def has_finite_verb(d) -> bool:
    return any(t.pos_ in {"VERB", "AUX"} and "Fin" in t.morph.get("VerbForm") for t in d)


# Berechne pro Schlagzeile: ohne finites Verb? Anteil der Substantive und Eigennamen?
corpus["verbless"] = [not has_finite_verb(d) for d in docs]
corpus["noun_share"] = [
    sum(t.pos_ in {"NOUN", "PROPN"} for t in d) / max(1, sum(t.pos_ != "PUNCT" for t in d)) for d in docs
]
# Mittelwerte pro Zeitung, als Prozent formatiert
style = corpus.groupby("outlet")[["verbless", "noun_share"]].mean()
style.style.format("{:.0%}")

Der Anteil der Schlagzeilen **ganz ohne finites Verb** ist hoch und unterscheidet sich zwischen den Zeitungen: Bei der SZ kommt mehr als jede zweite Schlagzeile ohne finites Verb aus, bei Der Standard ein kleinerer Anteil. Ein Grund ist das Format aus Skript 01: Die SZ setzt vor viele Schlagzeilen eine Dachzeile mit Gedankenstrich (*"Energiewende - Drunter und drüber beim Leitungsbau"*), und was danach folgt, ist oft eine Nominalphrase statt eines Satzes. Der Anteil der Substantive und Eigennamen liegt dagegen bei beiden Zeitungen nahe bei der Hälfte aller Wörter. Das ist eine Aussage über redaktionellen Stil, die mit den Werkzeugen aus Skript 01 nicht messbar gewesen wäre, weil "finites Verb" eine grammatische Kategorie ist und kein Muster in einer Zeichenkette.

Hier einige Schlagzeilen ohne Verb, um zu prüfen, ob das Maß das bedeutet, was wir glauben:

In [ ]:
# Zeige eine Stichprobe von Schlagzeilen ohne finites Verb
corpus.loc[corpus["verbless"], ["outlet", "title"]].sample(8, random_state=1)

Die Stichprobe zeigt, was "ohne finites Verb" in der Praxis heißt: Nominalphrasen (*Sondergipfel zu Griechenland*), Dachzeile plus Nominalphrase (*Energiewende - Drunter und drüber beim Leitungsbau*) und Einwortschlagzeilen (*Antennenfernsehen*). Das Maß misst also tatsächlich den Nominalstil. Dass alle acht Beispiele aus der SZ stammen, liegt an der Stichprobe: Die SZ stellt den Großteil des Korpus und hat den höheren Anteil verbloser Schlagzeilen.

### Nach Wortart filtern statt nach Stoppwortliste

Eine Stoppwortliste benennt, was *entfernt* werden soll. Mit POS-Tags können wir benennen, was wir *behalten* wollen. Für die Frage "Worum geht es in diesem Korpus?" tragen Substantive und Eigennamen den größten Teil des Signals:

In [ ]:
# Behalte pro Schlagzeile nur die Lemmata von Substantiven und Eigennamen
corpus["nouns"] = [
    [t.lemma_.casefold() for t in d if t.pos_ in {"NOUN", "PROPN"} and is_content(t)] for d in docs
]
# Zeige die 12 häufigsten Substantive pro Zeitung
pd.DataFrame(
    {
        outlet: [f"{w} ({n})" for w, n in Counter(w for ws in g["nouns"] for w in ws).most_common(12)]
        for outlet, g in corpus.groupby("outlet")
    }
)

Vergleiche das mit der Tabelle der häufigsten Terme in Skript 01: Das Schlagzeilen-Mobiliar (`neue`, `mehr`, `ab`, `wegen`, `warum`) ist verschwunden, ohne dass es jemand auflisten musste.

Der Filter ist für *diese* Frage richtig und für andere falsch. Wer Framing, Sentiment oder Zuschreibungen untersucht, braucht genau die Verben und Adjektive, die er verwirft.

## Dependenzparsing: Wer tut was

Das **Dependenzparsing** verbindet jedes Wort mit dem Wort, das es näher bestimmt, und benennt die Beziehung. Der Baum für die Beispielschlagzeile:

In [ ]:
# Zeichne den Dependenzbaum der Beispielschlagzeile
displacy.render(doc, style="dep", jupyter=True, options={"compact": True, "distance": 95})

Die deutschen Modelle von spaCy verwenden die Labels der TIGER-Baumbank: `sb` ist das Subjekt, `oa` das Akkusativobjekt (direktes Objekt), `mo` ein Modifikator, `nk` ein Element des Nominalkerns, `ROOT` das Hauptverb.

Diese Struktur ermöglicht einen klassischen Schritt der Informationsextraktion: Wir ziehen Tripel aus **Subjekt - Verb - Objekt** aus jeder Schlagzeile heraus. Hier eine bewusst einfache Version: Für jedes Verb nehmen wir sein Subjekt und, falls vorhanden, sein direktes Objekt.

In [ ]:
# Liefere für jedes Verb ein Tripel aus Subjekt, Verb und (falls vorhanden) Akkusativobjekt
def triples(d):
    for verb in d:
        if verb.pos_ not in {"VERB", "AUX"}:
            continue
        subjects = [c for c in verb.children if c.dep_ == "sb"]
        objects = [c for c in verb.children if c.dep_ == "oa"]
        for s in subjects:
            yield {
                "subject": s.lemma_,
                "verb": verb.lemma_,
                "object": objects[0].lemma_ if objects else "",
            }


# Sammle die Tripel aller Schlagzeilen in einer Tabelle
rows = [dict(t, outlet=o, title=d.text) for o, d in zip(corpus["outlet"], docs) for t in triples(d)]
svo = pd.DataFrame(rows)
print(f"{len(svo):,} Subjekt-Verb-Paare aus {svo['title'].nunique():,} Schlagzeilen")
svo.sample(8, random_state=2)

Aggregiert beantworten die Tripel die Frage, *wer laut Berichterstattung was tut*. Die Verben, die mit den häufigsten Eigennamen in Subjektposition verbunden sind:

In [ ]:
# Bestimme die acht häufigsten Subjekte, die Eigennamen sind
propn = {t.lemma_ for d in docs for t in d if t.pos_ == "PROPN"}
top_actors = svo[svo["subject"].isin(propn)]["subject"].value_counts().head(8).index
# Zeige pro Subjekt die fünf häufigsten Verben
(
    svo[svo["subject"].isin(top_actors)]
    .groupby("subject")["verb"]
    .agg(lambda v: ", ".join(f"{w} ({n})" for w, n in Counter(v).most_common(5)))
    .to_frame("häufigste Verben")
)

Die Akteure spiegeln das Korpus: Vereine und Ligen aus dem großen Sportteil der SZ (*Bayern* besiegt, gewinnt, schlägt), Staaten und Institutionen, die etwas *wollen*, und die Politikerinnen und Politiker des Jahres. Eine Zeile gehört nicht in die Liste: `der` ist kein Eigenname. Das Modell hat das Wort in einigen Schlagzeilen als Eigennamen markiert, und weil wir die Menge der Eigennamen aus genau diesen Markierungen gebildet haben, schlüpft der häufigste Artikel des Deutschen durch.

Diese Art von Analyse macht aus einem Haufen Schlagzeilen strukturierte Daten. Sie ist aber auch fragil: Schlagzeilen lassen das Verb oft weg (siehe Abschnitt "Wortarten: Was für eine Sprache ist eine Schlagzeile?") oder geben Zitate mit einem Doppelpunkt statt mit einem Verb wieder (*"Merkel: …"*), und der Parser macht Fehler bei Text im Telegrammstil, auf den er nicht trainiert wurde. Betrachte das Ergebnis als **Stichprobe verlässlicher Fälle**, nicht als vollständiges Verzeichnis.

## Named Entities

Die Komponente `ner` kennzeichnet Spans als `PER` (Person), `LOC` (Ort), `ORG` (Organisation) oder `MISC` (Sonstiges). Mit der Erkennung von Named Entities (Eigennamen) verschiebt sich die Frage davon, *welche Wörter* eine Zeitung verwendet, dazu, *über welche Akteure und Orte* sie berichtet.

In [ ]:
# Sammle alle Entities mit Zeitung und Label und zähle sie pro Label
ents = pd.DataFrame(
    [{"outlet": o, "text": e.text, "label": e.label_} for o, d in zip(corpus["outlet"], docs) for e in d.ents]
)
ents["label"].value_counts()

Das Modell findet in den Schlagzeilen Zehntausende Entities, am häufigsten Orte (`LOC`), dann Personen, sonstige Namen (`MISC`, etwa Ereignisse, Werke und Produkte) und Organisationen. Die vier Klassen sind ähnlich häufig; Schlagzeilen bestehen zu einem großen Teil aus Namen.

In [ ]:
# Zeige die n häufigsten Entities eines Labels pro Zeitung
def top_entities(label: str, n: int = 10) -> pd.DataFrame:
    sub = ents[ents["label"] == label]
    return pd.DataFrame(
        {
            outlet: [f"{t} ({c})" for t, c in g["text"].value_counts().head(n).items()]
            for outlet, g in sub.groupby("outlet")
        }
    )


# Orte
top_entities("LOC")

Die Orte: Spalte für Spalte die zehn häufigsten Orts-Entities je Zeitung mit ihrer Häufigkeit. Die Deutung folgt nach der Tabelle der Personen.

In [ ]:
# Personen
top_entities("PER")

Die Tabelle der Orte liest sich wie eine Landkarte der Prioritäten jeder Redaktion: Österreich, Wien und Salzburg in Der Standard, Berlin und Bayern in der SZ, dazu bei beiden die Schauplätze des Jahres - Griechenland, Syrien, die Türkei. Bei der Tabelle der Personen ist Vorsicht geboten. Ganz oben in der Standard-Spalte steht `Radiotipps`: die tägliche Fernsehrubrik aus Skript 01, deren Titel *TV- und Radiotipps für Sonntag, …* das Modell als Personennamen liest. Und dieselbe Person erscheint in zwei Zeilen (`Trump` und `Donald Trump`), weil die Erkennung nur Zeichenketten liefert und nicht weiß, wer gemeint ist.

### Wie weit wir den Entities trauen können

Statistische Named-Entity-Erkennung wird auf Fließtext trainiert, meist auf Nachrichtenartikeln, in denen Personen mit vollem Namen eingeführt werden. Schlagzeilen verwenden bloße Nachnamen, und das Modell muss aus dem Kontext raten. Vergleiche die beiden Modellgrößen anhand einiger Schlagzeilen:

In [ ]:
# Lade zum Vergleich das kleine Modell
nlp_sm = spacy.load("de_core_news_sm")
# Wähle Schlagzeilen, die mit "Name: " beginnen
samples = corpus.loc[corpus["title"].str.contains(r"^\w+: ", regex=True), "title"].sample(6, random_state=4)
# Stelle die Entities beider Modelle nebeneinander
pd.DataFrame(
    {
        "Schlagzeile": samples.values,
        "sm": [", ".join(f"{e.text}/{e.label_}" for e in nlp_sm(s).ents) for s in samples],
        "md": [", ".join(f"{e.text}/{e.label_}" for e in nlp(s).ents) for s in samples],
    }
)

Aus diesem Vergleich lernen wir zweierlei:

- **Modelle sind sich uneinig, und beide machen Fehler.** Dieselben Spans bekommen verschiedene Labels (*TTIP* ist einmal `MISC`, einmal `ORG`), und gewöhnliche Wörter werden zu Entities (*Registrierkasse* als Person, *Waffenruhe* als Ort). Aggregierte Häufigkeiten sind vertrauenswürdiger als jedes einzelne Label, weil Fehler verstreut auftreten, während echte Muster sich wiederholen.
- **Prüfe, bevor du darauf aufbaust.** Bevor du berichtest "Zeitung X erwähnt Person Y am häufigsten", lies eine Stichprobe der gefundenen Schlagzeilen. Wenn Entities für eine Fragestellung zentral sind, lohnt sich ein größeres Modell (`de_core_news_lg`) oder ein Transformer-basiertes; für einen schnellen Überblick reicht `md`.

## Regelbasiertes Matching: eigenes Wissen hinzufügen

Statistische Modelle kennen allgemeines Deutsch. Oft wissen wir etwas Spezifischeres - hier: die Namen der politischen Parteien in zwei Ländern -, und spaCy erlaubt es, dieses Wissen als **Regeln** hinzuzufügen. Ein `EntityRuler`, der vor der statistischen Komponente `ner` eingefügt wird, kennzeichnet seine Treffer zuerst, und `ner` respektiert sie.

In [ ]:
# Nur Parteien, die es in genau einem Land gibt. Die Grünen gibt es in beiden Ländern, und "Linke"
# ist auch das gewöhnliche Wort für "die Linke" - für eine Zählung nach Ländern taugt keines von beiden.
PARTIES = {
    "AT": ["ÖVP", "SPÖ", "FPÖ", "NEOS"],
    "DE": ["CDU", "CSU", "SPD", "AfD", "FDP"],
}
# Ordne jeder Partei ihr Land zu
party_country = {p: c for c, ps in PARTIES.items() for p in ps}

# Lade eine weitere Kopie des Modells und füge den EntityRuler vor ner ein
nlp_parties = spacy.load("de_core_news_md")
ruler = nlp_parties.add_pipe("entity_ruler", before="ner")
ruler.add_patterns([{"label": "PARTY", "pattern": p} for p in party_country])
# Parteien kommen auch als erster Teil eines Kompositums mit Bindestrich vor: "CDU-Chef".
ruler.add_patterns(
    [{"label": "PARTY", "pattern": [{"TEXT": {"REGEX": rf"^{p}-"}}]} for p in party_country]
)

# Teste die Regeln an einem Beispielsatz
print([(e.text, e.label_) for e in nlp_parties("FPÖ-Chef Strache kritisiert ÖVP, die CDU schweigt").ents])

Ein Regex-Muster auf den Text des Tokens erfasst die Formen mit Bindestrich. Für die Zählung ordnen wir jeden Treffer wieder seiner Partei zu:

In [ ]:
# Annotiere alle Schlagzeilen mit der erweiterten Pipeline (Parser und Lemmatizer werden nicht gebraucht)
party_docs = list(nlp_parties.pipe(corpus["title"], disable=["parser", "lemmatizer"]))


# Führe einen Treffer auf seine Partei zurück: "CDU-Chef" -> "CDU"
def party_of(span_text: str) -> str:
    return span_text.split("-")[0]


# Sammle alle Parteinennungen mit Zeitung und Land der Zeitung
mentions = pd.DataFrame(
    [
        {"outlet": o, "outlet_country": c, "party": party_of(e.text)}
        for o, c, d in zip(corpus["outlet"], corpus["country"], party_docs)
        for e in d.ents
        if e.label_ == "PARTY"
    ]
)
# Kreuztabelle: Zeitung x Land der Partei
mentions["party_country"] = mentions["party"].map(party_country)
pd.crosstab(mentions["outlet"], mentions["party_country"], margins=True)

Beide Zeitungen schreiben ganz überwiegend über die Parteien des eigenen Landes, aber die Tabelle ist **asymmetrisch**: Der anteilige Blick über die Grenze ist bei Der Standard mehr als doppelt so groß wie bei der SZ. Die österreichische Zeitung beobachtet die deutsche Politik genauer als umgekehrt - ein bekanntes Muster zwischen einem kleinen und einem großen Nachbarland mit gemeinsamer Sprache. In Skript 03 messen wir solche Unterschiede systematisch.

Regeln sind präzise und transparent, aber sie finden nur, was wir auflisten, und die Wahl der Liste ist selbst eine analytische Entscheidung. Die erste Version dieses Regelwerks enthielt die Grünen und *Die Linke* und ordnete sie anhand der Schreibweise einem Land zu - `Grüne` Österreich, `Grünen` Deutschland. Das ist aber Flexion, nicht Nationalität: Beide Länder haben eine grüne Partei, und *Linke* bezeichnet auch ganz allgemein "die Linke". Das Lesen der gefundenen Schlagzeilen hat den Fehler sofort aufgedeckt. Regeln und statistische Modelle ergänzen einander: Regeln für geschlossene Mengen, die sich eindeutig aufzählen lassen, Modelle für offene Klassen wie Personennamen.

## Eine Anmerkung zur Sprache

Alles bisher setzt deutschen Input voraus. Unser Korpus ist deutschsprachig, aber nicht jedes Korpus ist es, und auch hier stecken englische Film-, Produkt- und Firmennamen in den Schlagzeilen. Was passiert, wenn das deutsche Modell auf eine englische Schlagzeile trifft?

In [ ]:
# Wende das deutsche Modell auf eine englische Schlagzeile an
english = "Mario Draghi: The outlook for the euro area economy"
print([(t.text, t.lemma_, t.pos_) for t in nlp(english)])

Das deutsche Modell produziert selbstbewussten Unsinn: Da es englische Funktionswörter nicht erkennt, kennzeichnet es fast jedes Wort als Eigennamen oder lässt es unklassifiziert, ohne jeden Hinweis darauf, dass etwas nicht stimmt. Mehrsprachige Korpora brauchen zuerst eine Spracherkennung und dann ein Modell pro Sprache - spaCys `en_core_web_lg` etwa für die englischen Texte. Skript 06 zeigt mit mehrsprachigen Embeddings einen anderen Weg.

## Zusammenfassung

| Annotation | ersetzt aus Skript 01 | verwenden für |
|---|---|---|
| Lemma | nichts - Skript 01 konnte das nicht | flektierte Formen zusammenführen |
| `is_stop` | die Stoppwortliste von NLTK | Funktionswörter entfernen |
| Wortart (POS) | Casefolding als Substantiv-Detektor | Inhaltswörter behalten; Stilmaße |
| Dependenzparsing | - | Extraktion von "wer tut was" |
| Named Entities | - | Akteure und Orte |
| Regeln (`EntityRuler`) | handgeschriebene Regexes über Zeichenketten | Wissen über die Domäne, das sich aufzählen lässt |

Das Modell hat Information hinzugefügt und mit ihr eine neue Art von Fehler: Annotationsfehler, die autoritativ wirken. Die Gewohnheit, die wir uns aneignen sollten: **Stichproben dessen lesen, was das Modell produziert hat**, bevor wir es aggregieren.

## Übungen

1. **Stil im Zeitverlauf.** Ist der Anteil der Schlagzeilen ohne Verb für jede Zeitung von Monat zu Monat stabil, oder verschiebt er sich? Gruppiere nach `corpus["month"]` (die SZ-Daten kennen nur den Monat). Unterscheidet sich der Anteil zwischen den Ressortgruppen (`section_group`) stärker als zwischen den Zeitungen?
2. **Zuschreibung von Zitaten.** Viele Schlagzeilen beginnen mit `Name: "…"`. Schreibe ein Muster für den `Matcher` von spaCy, das einen Eigennamen gefolgt von einem Doppelpunkt am Anfang der Schlagzeile findet, und zähle die am häufigsten zitierten Personen pro Zeitung.
3. **Fehler bei Entities.** Nimm die 30 häufigsten `PER`-Entities und prüfe jede von Hand. Welcher Anteil sind wirklich Personen? Welche Fehler sind systematisch?
4. **Morphologie.** Verwende `token.morph.get("Number")`, um herauszufinden, welche Substantive überwiegend im Plural vorkommen. Sagt das Ergebnis etwas darüber aus, wie Nachrichten gerahmt werden (*Tote*, *Angriffe*, *Proteste*)?

## Quellen

* https://spacy.io/models/de
* https://spacy.io/usage/linguistic-features
* https://spacy.io/usage/rule-based-matching
* https://pandas.pydata.org/docs/